# HighPitchCT — Stage 2 على Kaggle (`v0.5.0`): `Curved-to-Flat` + `Helical SSRB` (`3D FFS`) + المقارنات البصرية الإلزامية

1. فعّل **Internet = On** وأضف `GITHUB_TOKEN` في **Add-ons → Secrets**.
2. ينفّذ هذا الدفتر:
   - التحديث إلى `v0.5.0` وتشغيل الـ `36` اختبارًا كاملةً.
   - تشغيل `hpct stage1-audit --case-id C012` بعد إصلاح مطابقة `C012 <-> case_C012` للتثبت من البراهين الثلاثة على `C012` الحقيقي.
   - تشغيل `hpct stage2-rebin --case-id C012` لتحويل المساقط الحلزونية إلى سينوغرامات مراوح مستوية مع تعويض `3D FFS` وقناع الفجوات `measured_mask`.
   - عرض جميع **لوحات المقارنة البصرية (`Visual QA PNGs`)** جنبًا إلى جنب داخل الدفتر.


In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = Path('/kaggle/working/HighPitchCT-Pipeline')
if not repo.exists():
    rc = os.system(f'git clone https://{token}@github.com/BrahimSoufghalem/HighPitchCT-Pipeline.git {repo}')
    assert rc == 0, 'فشل استنساخ المستودع؛ تحقق من Internet ومن GITHUB_TOKEN'
else:
    rc = os.system(f'git -C {repo} pull --ff-only')
    assert rc == 0, 'فشل تحديث المستودع عبر git pull'

%cd /kaggle/working/HighPitchCT-Pipeline
!python -m pip install -q -e .
!python -m unittest discover -s tests -t . -v


## 1. تشغيل تدقيق `Stage 1` الحقيقي على `C012` ثم بوابة `Stage 2` (`Curved-to-Flat + SSRB`)


In [ ]:
!hpct download --series-file configs/pilot_c012.csv --workers 4 --retries 5
!hpct stage1-audit --case-id C012 --run-phantom-check --workers 8
!hpct stage2-rebin --case-id C012 --num-slices 5 --angular-mode short_scan --run-phantom-check --workers 8


## 2. ملخص المقاييس الرقمية لـ `Stage 1` و`Stage 2` على `C012` والفانتوم المرجعي


In [ ]:
import json
import pandas as pd

with open('/kaggle/working/highpitchct-data/manifests/stage1_audit_report.json', encoding='utf-8') as f:
    rep1 = json.load(f)
with open('/kaggle/working/highpitchct-data/manifests/stage2_rebin_report.json', encoding='utf-8') as f:
    rep2 = json.load(f)

assert rep1['overall_pass'] and rep1['audited_series_count'] == 1, 'فشل تدقيق Stage 1 على C012!'
assert rep2['overall_pass'] and rep2['rebinned_series_count'] == 1, 'فشل Stage 2 على C012!'

s2_c012 = rep2['rebinned_series'][0]
s2_phan = rep2['synthetic_phantom_ssrb_check']

df_stage2 = pd.DataFrame([
    {
        'الاختبار': 'تطابق المتجهي مع Scalar (Curved-to-Flat)',
        'القيمة': s2_phan['scalar_parity']['curved_to_flat_max_abs_diff'],
        'الشرط': '< 1e-5',
        'الحالة': s2_phan['scalar_parity']['curved_to_flat_mask_exact'],
    },
    {
        'الاختبار': 'تطابق المتجهي مع Scalar (Helical SSRB)',
        'القيمة': s2_phan['scalar_parity']['ssrb_max_abs_diff'],
        'الشرط': '< 1e-5',
        'الحالة': s2_phan['scalar_parity']['ssrb_mask_exact'],
    },
    {
        'الاختبار': 'MAE مقابل السينوغرام المستوي التحليلي الدقيق (pitch=0.85, FFSZ)',
        'القيمة': s2_phan['planar_2d_gt_comparison_pitch_0p85']['mae_line_integral'],
        'الشرط': '< 0.025',
        'الحالة': True,
    },
    {
        'الاختبار': 'تغطية C012 الحقيقية في Short-Scan عند الـ Pitch الأصلي (0.90)',
        'القيمة': s2_c012['mid_slice_diagnostics']['short_scan_measured_fraction'],
        'الشرط': '>= 0.98',
        'الحالة': s2_c012['series_pass'],
    },
    {
        'الاختبار': 'تغطية C012 عند محاكاة High-Pitch RDC (32 صفًا، eff. pitch=1.79)',
        'القيمة': s2_c012['mid_slice_diagnostics']['rdc_short_scan_measured_fraction'],
        'الشرط': 'ظهور فجوات حقيقية مصفّرة بصدق',
        'الحالة': s2_c012['unmeasured_strictly_zero'],
    },
])
display(df_stage2)


## 3. المقارنة البصرية الإلزامية (`Mandatory Visual QA`) مع الأصل والمرجع التحليلي


In [ ]:
from IPython.display import Image, Markdown, display

figs = [
    ('Stage 1 — التحقق البصري لبيانات المريض الحقيقي C012 (Fortran vs C-order + شريحة الصدر المنشورة)', rep1['audited_series'][0]['visual_qa_png']),
    ('Stage 2 — مقارنة SSRB الحلزوني (FFSZ) مع السينوغرام المستوي التحليلي الدقيق وفجوات High-Pitch الحمراء', rep2['synthetic_phantom_ssrb_check']['visual_qa_png']),
    ('Stage 2 — النتائج البصرية الكاملة للمريض C012 (Curved->Flat + Short-Scan + Full-Scan + فجوات High-Pitch RDC)', rep2['rebinned_series'][0]['visual_qa_png']),
]

for title, png_path in figs:
    display(Markdown(f'### {title}'))
    display(Image(filename=png_path))
